# Keyword Detection (TensorFlow)
This notebook contains the TensorFlow implementation of the Keyword Detection model with GPU support.

## Loading Requirements

In [29]:
import wave
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np

from sklearn.model_selection import train_test_split

## 0. Check GPU Availability


In [30]:
def check_gpu_status():
    gpus = tf.config.list_physical_devices('GPU')
    print(f"CUDA Available: {len(gpus) > 0}")
    if gpus:
        print(f"Number of GPUs detected: {len(gpus)}\n")
        for i, gpu in enumerate(gpus):
            print(f"--- Available GPUs: {i} ---")
            print(f"Name: {gpu.name}")
    else:
        print("TensorFlow cannot detect a compatible GPU. It will default to CPU.")

check_gpu_status()

CUDA Available: False
TensorFlow cannot detect a compatible GPU. It will default to CPU.


## 1. Get Audio File Specifications


In [31]:
def check_wav_specs_tf(file_path):
    if not os.path.exists(file_path):
        print(f"File not found: {file_path}")
        return
    
    audio_bin = tf.io.read_file(file_path)

    audio, sample_rate = tf.audio.decode_wav(audio_bin, desired_channels=-1)
    
    # audio shape is [samples, channels]
    num_samples = tf.shape(audio)[0].numpy()
    num_channels = tf.shape(audio)[1].numpy()
    sample_rate = sample_rate.numpy()
    duration = num_samples / sample_rate

    print(f"Sample Rate: {sample_rate} Hz")
    print(f"Channels: {num_channels}")
    print(f"Frames (Samples): {num_samples}")
    print(f"Duration: {duration:.2f} seconds")
    
check_wav_specs_tf("../dataset/yes/1a673010_nohash_1.wav")

Sample Rate: 16000 Hz
Channels: 1
Frames (Samples): 16000
Duration: 1.00 seconds


## 2. Data Preparation

In [32]:
import json
import os
from sklearn.model_selection import train_test_split

path_to_config = os.path.join('..', 'config.json')

# 1. Konfiguration laden
try:
    with open(path_to_config, 'r') as f:
        config = json.load(f)
    print("Konfiguration erfolgreich geladen!")
except FileNotFoundError:
    print(f"Fehler: Datei nicht gefunden unter {os.path.abspath(path_to_config)}")
    print(f"Aktuelles Verzeichnis: {os.getcwd()}")
    raise

# 2. Variablen setzen (Wichtig: DATA_DIR muss evtl. auch angepasst werden)
CLASSES = config["keywords"]
# Wenn DATA_DIR in der Config "dataset/" ist, liegt dieser Ordner vermutlich
# auch im Hauptverzeichnis. Also setzen wir auch hier ".." davor.
DATA_DIR = os.path.join('..', config["DATA_DIR"])
NUM_SAMPLES = config["num_samples"]
TARGET_SAMPLE_RATE = config["target_sample_rate"]
BATCH_SIZE = 32

# 3. Funktion zum Einlesen
def get_files_and_labels():
    file_paths = []
    labels = []
    class_to_idx = {cls_name: i for i, cls_name in enumerate(CLASSES)}

    for cls_name in CLASSES:
        cls_dir = os.path.join(DATA_DIR, cls_name)
        if not os.path.exists(cls_dir):
            print(f"Warnung: Verzeichnis {cls_dir} nicht gefunden.")
            continue
        for file in os.listdir(cls_dir):
            if file.endswith(".wav"):
                file_paths.append(os.path.join(cls_dir, file))
                labels.append(class_to_idx[cls_name])
    return file_paths, labels

# 4. Daten laden und splitten
file_paths, labels = get_files_and_labels()
if len(file_paths) > 0:
    train_paths, test_paths, train_labels, test_labels = train_test_split(
        file_paths, labels, test_size=0.2, stratify=labels, random_state=42
    )
    print(f"Erfolg: {len(file_paths)} Dateien gefunden.")
else:
    print(f"Keine Dateien gefunden! Gesuchter Pfad: {os.path.abspath(DATA_DIR)}")

Konfiguration erfolgreich geladen!
Erfolg: 9486 Dateien gefunden.


## 3. Dataset Pipeline (TensorFlow native audio layer)

In [33]:
@tf.function
def process_path(file_path, label):
    # Read and decode wav
    audio_binary = tf.io.read_file(file_path)
    audio, _ = tf.audio.decode_wav(audio_binary, desired_channels=1)
    
    audio = tf.squeeze(audio, axis=-1)
    
    # Pad or cut to exactly 1 second (16000 samples)
    audio_length = tf.shape(audio)[0]
    
    # Pad
    padding = tf.maximum(NUM_SAMPLES - audio_length, 0)
    # audio will be padded at the end
    audio = tf.pad(audio, [[0, padding]])
    
    # Slice
    audio = audio[:NUM_SAMPLES]
    
    return audio, label

def create_dataset(paths, labels, batch_size, shuffle=False):
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        dataset = dataset.shuffle(buffer_size=len(paths))
    dataset = dataset.map(process_path, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

train_ds = create_dataset(train_paths, train_labels, BATCH_SIZE, shuffle=True)
test_ds = create_dataset(test_paths, test_labels, BATCH_SIZE, shuffle=False)

## 4. Model Architecture (with onboard MelSpectrogram on GPU)

In [34]:
# Mel spectrogram parameters mapping to 64 mels equivalent
frame_length = 1024
frame_step = 512
fft_length = 1024
num_mel_bins = 64
lower_edge_hertz = 80.0
upper_edge_hertz = 7600.0

class MelSpectrogram(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(MelSpectrogram, self).__init__(**kwargs)

    def call(self, audio):
        # audio shape: (batch_size, num_samples)
        stfts = tf.signal.stft(audio, frame_length=frame_length,
                               frame_step=frame_step, fft_length=fft_length)
        spectrograms = tf.abs(stfts)
        
        # Warp the linear scale spectrograms into the mel-scale.
        num_spectrogram_bins = stfts.shape[-1]
        mel_weight_matrix = tf.signal.linear_to_mel_weight_matrix(
            num_mel_bins, num_spectrogram_bins, TARGET_SAMPLE_RATE,
            lower_edge_hertz, upper_edge_hertz)
        
        mel_spectrograms = tf.tensordot(spectrograms, mel_weight_matrix, 1)
        mel_spectrograms.set_shape(spectrograms.shape[:-1].concatenate(
            mel_weight_matrix.shape[-1:]))
            
        # Log mel spectrograms
        log_mel_spectrograms = tf.math.log(mel_spectrograms + 1e-6)
        
        # Add channel dimension
        return tf.expand_dims(log_mel_spectrograms, -1)

def build_model(num_classes):
    audio_input = tf.keras.Input(shape=(NUM_SAMPLES,), name='audio')
    
    x = MelSpectrogram()(audio_input)


    x = tf.keras.layers.Conv2D(16, 3, padding='same', activation='relu')(x)
    x = tf.keras.layers.MaxPooling2D(2, 2)(x)
    
    x = tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu')(x)
    x = tf.keras.layers.MaxPooling2D(2, 2)(x)
    
    x = tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = tf.keras.layers.MaxPooling2D(2, 2)(x)
    
    x = tf.keras.layers.Resizing(4, 4)(x)
    
    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(128, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(num_classes)(x)
    
    return tf.keras.Model(inputs=audio_input, outputs=outputs)

model = build_model(len(CLASSES))
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ audio (InputLayer)              │ (None, 16000)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mel_spectrogram                 │ (None, 30, 64, 1)      │             0 │
│ (MelSpectrogram)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 30, 64, 16)     │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 15, 32, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 15, 32, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 7, 16, 32)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 7, 16, 64)      │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 3, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resizing (Resizing)             │ (None, 4, 4, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       131,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 155,012 (605.52 KB)

 Trainable params: 155,012 (605.52 KB)

 Non-trainable params: 0 (0.00 B)

## 5. Training Loop

In [35]:
optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

model.compile(optimizer=optimizer,
              loss=loss_fn,
              metrics=['accuracy'])

print("Starting training (TensorFlow automatically uses GPU if detected)...")
history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=EPOCHS
)

NameError: name 'LEARNING_RATE' is not defined

## 6. Visualization

In [ ]:
file_path = '../dataset/yes/00f0204f_nohash_0.wav'
if os.path.exists(file_path):
    audio_binary = tf.io.read_file(file_path)
    audio, _ = tf.audio.decode_wav(audio_binary, desired_channels=1)
    audio = tf.squeeze(audio, axis=-1)

    # STFT
    stfts = tf.signal.stft(audio, frame_length=1024, frame_step=512, fft_length=1024)
    spectrograms = tf.abs(stfts)
            
    # Mel Spectrogram (using 128 bins like in original plot code)
    mel_weight_matrix = tf.signal.linear_to_mel_weight_matrix(
        128, spectrograms.shape[-1], 16000, 80.0, 7600.0)
    mel_spectrograms = tf.tensordot(spectrograms, mel_weight_matrix, 1)

    # Amplitude to DB (log)
    log_mel_spectrogram = 10.0 * tf.math.log(tf.maximum(mel_spectrograms, 1e-10)) / tf.math.log(10.0)

    plt.figure(figsize=(10, 4))
    plt.imshow(tf.transpose(log_mel_spectrogram).numpy(), cmap='inferno', origin='lower', aspect='auto')
    plt.title(f'Mel Spektrogramm fuer {os.path.basename(file_path)}')
    plt.xlabel('Frames (Zeit)')
    plt.ylabel('Mel-Frequenzbänder')
    plt.colorbar(format='%+2.0f dB')
    plt.tight_layout()
    plt.show()
else:
    print(f"Datei {file_path} nicht gefunden.")